# Bank Customer Churn & Retention Analytics — Machine Learning

This notebook builds and evaluates the Logistic Regression churn model used in the project.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

## 1. Load Dataset

In [ ]:
# Upload the prepared CSV in Google Colab, then set the filename below.
FILE_NAME = "Bank_Customer_Churn_SQL_Ready.csv"

df = pd.read_csv(FILE_NAME)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Basic checks
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate Customer IDs:", df["CustomerId"].duplicated().sum())
print("\nChurn distribution:")
print(df["Exited"].value_counts())

## 2. Prepare Features and Target

`Exited = 1` represents churn and `Exited = 0` represents retained customers. CustomerId is an identifier and is excluded from modelling.

In [ ]:
X = df.drop(columns=["Exited", "CustomerId"])
y = df["Exited"]

categorical_features = ["Geography", "Gender"]
numeric_features = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember",
    "EstimatedSalary"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

## 3. Stratified Train/Test Split

A stratified 80/20 split is used so that the churn class remains represented in both training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Churners in test set:", int(y_test.sum()))

## 4. Logistic Regression Model

`class_weight='balanced'` is used because churn is the minority class.

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=1000
        ))
    ]
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

## 5. Model Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

display(results)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 6. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Customer Churn Prediction")
plt.legend()
plt.show()

## 7. Threshold Analysis

The project examines thresholds from 0.30 to 0.70. A threshold of 0.40 is used as a practical prioritization threshold.

In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
rows = []

for threshold in thresholds:
    pred_threshold = (y_prob >= threshold).astype(int)
    caught = int(((pred_threshold == 1) & (y_test.values == 1)).sum())
    false_alarms = int(((pred_threshold == 1) & (y_test.values == 0)).sum())

    rows.append({
        "Threshold": threshold,
        "Churners Caught": f"{caught} / {int(y_test.sum())}",
        "Recall": recall_score(y_test, pred_threshold, zero_division=0),
        "Precision": precision_score(y_test, pred_threshold, zero_division=0),
        "False Alarms": false_alarms
    })

threshold_df = pd.DataFrame(rows)
display(threshold_df)

## 8. Customer Risk Scoring

Risk bands are created from predicted churn probabilities for the held-out test customers.

In [ ]:
risk_df = X_test.copy()
risk_df["CustomerId"] = df.loc[X_test.index, "CustomerId"]
risk_df["Actual_Exited"] = y_test
risk_df["Churn_Probability"] = y_prob

risk_df["Risk_Band"] = pd.cut(
    risk_df["Churn_Probability"],
    bins=[-np.inf, 0.30, 0.60, np.inf],
    labels=["Low", "Medium", "High"]
)

risk_summary = (
    risk_df.groupby("Risk_Band", observed=False)
    .agg(
        Customers=("CustomerId", "count"),
        Churned=("Actual_Exited", "sum")
    )
)

risk_summary["Actual_Churn_Rate"] = (
    risk_summary["Churned"] / risk_summary["Customers"] * 100
)

display(risk_summary)
display(risk_df.head())

## 9. Export Predictions

The prediction file can be uploaded to the `05_ML_Modelling` GitHub folder.

In [ ]:
output = risk_df[
    ["CustomerId", "Actual_Exited", "Churn_Probability", "Risk_Band"]
].copy()

output.to_csv("customer_churn_predictions.csv", index=False)
print("Saved: customer_churn_predictions.csv")

## Conclusion

The Logistic Regression model is used as a customer-retention prioritization tool. The model estimates churn probability and groups customers into risk bands; it should not be interpreted as a definitive statement that a particular customer will leave.